# `benchmark.py`启动命令说明

In [ ]:
%%bash
cd /data/wanghanzhen/Projects/MTP/NIPS26/dflash

$WHZ_DIR/models/z-lab/Qwen3-8B-DFlash-b16
$WHZ_DIR/models/Qwen/Qwen3-8B

CUDA_VISIBLE_DEVICES=5 python3 benchmark.py \
  --dataset gsm8k \
  --target-model $WHZ_DIR/models/Qwen/Qwen3-8B \
  --draft-model $WHZ_DIR/models/z-lab/Qwen3-8B-DFlash-b16 \

# `benchmark_sglang.py` 启动命令说明

这个 notebook 记录 `/data/wanghanzhen/Projects/MTP/NIPS26/dflash/benchmark_sglang.py` 的关键启动参数、样本数计算方式、长度控制方式和常用命令示例。

脚本会自动启动 SGLang server，先跑 baseline，再跑 DFLASH；如果加 `--skip-baseline`，只跑 DFLASH。每个 attention backend 都会单独启动服务并遍历并发数。

## 关键参数

| 参数 | 默认值 | 说明 |
| --- | --- | --- |
| `--dataset-name` | `gsm8k` | 数据集名。若 `dataset_path.json` 中存在同名 key，会映射到对应 jsonl 文件。当前可用映射包括 `passkey`、`number_string`、`kv_retrieval`、`longbook_sum_eng`、`longbook_choice_eng`、`longbook_qa_eng`、`longbook_qa_chn`、`longdialogue_qa_eng`、`math_find`、`math_calc`、`code_run`、`code_debug`、`2wikimqa_e`。 |
| `--concurrencies` | `1,2,4,8,16,32` | 并发 sweep 列表，逗号分隔。 |
| `--questions-per-concurrency-base` | `128` | 每个并发下样本数基数。实际计入指标的样本数 `n = min(base * concurrency, max_questions_per_config)`。 |
| `--max-questions-per-config` | `1024` | 单个 `(backend, concurrency)` 配置下计入指标的最大样本数。 |
| `--batch-requests` | 关闭 | 使用 server-side batched `/generate`，batch size 等于 concurrency；否则使用 client-side 并发请求。 |
| `--attention-backends` | `flashinfer,fa3,fa4` | attention backend 列表。脚本会自动跳过不适配当前 GPU 架构的 `fa3`/`fa4`。 |
| `--tp-size` | `1` | Tensor parallel size，单值，不做 sweep。 |
| `--mem-fraction-static` | `0.75` | 传给 SGLang server 的静态显存比例。OOM 时可降低。 |
| `--max-running-requests` | `64` | SGLang server 最大运行请求数。通常应不小于最大 concurrency。 |


## 样本数和 warmup 规则

对每个并发 `c`，脚本使用：

```text
n = min(questions_per_concurrency_base * c, max_questions_per_config)
```

`n` 是计入指标的样本数。脚本会在每个配置正式计时前额外跑一个 warmup batch，大小等于当前并发 `c`，并从指标中丢弃。因此每个配置实际需要准备：

```text
required_prompts_for_this_concurrency = n + c
```

整个 sweep 会按最大并发预先准备：

```text
required_questions = max(n for c in concurrencies) + max(concurrencies)
```

默认并发 `1,2,4,8,16,32`、默认 `base=128`、默认 `max_questions_per_config=1024` 时：

| concurrency | 计入指标样本数 n | warmup 样本数 | 需要 prompt 数 |
| --- | ---: | ---: | ---: |
| 1 | 128 | 1 | 129 |
| 2 | 256 | 2 | 258 |
| 4 | 512 | 4 | 516 |
| 8 | 1024 | 8 | 1032 |
| 16 | 1024 | 16 | 1040 |
| 32 | 1024 | 32 | 1056 |

如果数据集条数不足，脚本会循环复用样本。

## 最大长度相关说明

- `--max-new-tokens` 控制输出长度上限，也就是每个 `/generate` 请求最多生成多少 token。
- `--context-length` 控制 SGLang server 的上下文长度上限，长上下文任务需要显式设置。
- `--max-input-tokens` 控制输入 prompt 过滤。超过该 token 数的 prompt 会跳过，不参与测试。
- 如果设置 `--context-length 200000` 但不设置 `--max-input-tokens`，脚本会使用 `max_input_tokens = 200000`。
- 对 InfiniteBench 这类长上下文 jsonl，建议同时设置 `--context-length`，并根据任务调整 `--max-new-tokens`。例如 retrieval 类可用较小输出长度，summarization 类需要更大输出长度。

## 命令示例 1：快速 smoke test

用途：确认环境、模型加载和 DFLASH sanity check 是否正常。只跑 DFLASH，少量样本，小生成长度。

In [ ]:
%%bash
cd /data/wanghanzhen/Projects/MTP/NIPS26/dflash

SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN=1 \
CUDA_VISIBLE_DEVICES=1 python /data/wanghanzhen/Projects/MTP/NIPS26/dflash/benchmark_sglang.py \
  --dataset-name longbench_v2 \
  --target-model $WHZ_DIR/models/Qwen/Qwen3-8B \
  --draft-model $WHZ_DIR/models/z-lab/Qwen3-8B-DFlash-b16 \
  --response-json response_longbench_v2_dflash_full.json \
  --context-length 60000

## 命令示例 2：标准短任务 sweep

用途：在 `gsm8k` 上跑 baseline + DFLASH，对并发 `1,2,4,8,16,32` 做吞吐对比。

In [ ]:
%%bash
cd /data/wanghanzhen/Projects/MTP/NIPS26/dflash

CUDA_VISIBLE_DEVICES=0 python3 benchmark_sglang.py \
  --dataset-name gsm8k \
  --target-model Qwen/Qwen3-8B \
  --draft-model z-lab/Qwen3-8B-DFlash-b16 \
  --attention-backends flashinfer \
  --concurrencies 1,2,4,8,16,32 \
  --questions-per-concurrency-base 128 \
  --max-questions-per-config 1024 \
  --max-new-tokens 2048 \
  --max-running-requests 64 \
  --response-json response_gsm8k.json \
  --output-md report_gsm8k.md

## 命令示例 3：长上下文 longbook_qa_eng

用途：跑 `passkey` 这类长上下文但短输出任务。`passkey` 会通过 `dataset_path.json` 映射到 InfiniteBench jsonl 文件。

In [ ]:
%%bash
cd /data/wanghanzhen/Projects/MTP/NIPS26/dflash

/data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP/cache/models/dflash_sample_40000_think_off_qwen3_8b_maxlen4096/epoch_6_step_29844

$WHZ_DIR/models/z-lab/Qwen3-8B-DFlash-b16
$WHZ_DIR/models/Qwen/Qwen3-8B

SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN=1 \
CUDA_VISIBLE_DEVICES=3,4 python3 benchmark_sglang.py \
  --dataset-name longbook_qa_eng \
  --target-model $WHZ_DIR/models/Qwen/Qwen3-8B \
  --draft-model  /data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP/cache/models/dflash_sample_40000_think_off_qwen3_8b_maxlen4096/epoch_6_step_29844 \
  --skip-baseline \
  --attention-backends flashinfer \
  --concurrencies 1 \
  --tp-size 2 \
  --questions-per-concurrency-base 10 \
  --max-questions-per-config 128 \
  --context-length 320000 \
  --max-input-tokens 240000 \
  --max-new-tokens 512 \
  --timeout-s 7200 \
  --response-json response_longbook_qa_eng_dflash_demo.json \
  --output-md report_longbook_qa_eng_demo.md

## 命令示例 4：长文本总结任务

用途：跑 `longbook_sum_eng`。总结任务输出较长，`--max-new-tokens` 可以比 retrieval 任务设大。若 OOM，可降低 `--concurrencies`、`--mem-fraction-static` 或 `--max-running-requests`。

In [ ]:
%%bash
cd /data/wanghanzhen/Projects/MTP/NIPS26/dflash

SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN=1 \
CUDA_VISIBLE_DEVICES=0,1 python3 benchmark_sglang.py \
  --dataset-name longbook_sum_eng \
  --target-model $WHZ_DIR/models/Qwen/Qwen3-8B \
  --draft-model $WHZ_DIR/models/z-lab/Qwen3-8B-DFlash-b16 \
  --skip-baseline \
  --attention-backends flashinfer \
  --concurrencies 1 \
  --tp-size 2 \
  --questions-per-concurrency-base 10 \
  --max-questions-per-config 10 \
  --context-length 320000 \
  --max-input-tokens 200000 \
  --max-new-tokens 2048 \
  --timeout-s 7200 \
  --mem-fraction-static 0.70 \
  --response-json response_longbook_sum_eng_dflash_full.json \
  --output-md report_longbook_sum_eng_dflash_full.md

## 命令示例 5：server-side batch 请求

用途：把同一并发下的 prompts 作为 batched `/generate` 发给 server。batch size 等于当前 concurrency。

In [ ]:
%%bash
cd /data/wanghanzhen/Projects/MTP/NIPS26/dflash

CUDA_VISIBLE_DEVICES=0 python3 benchmark_sglang.py \
  --dataset-name kv_retrieval \
  --target-model Qwen/Qwen3-8B \
  --draft-model z-lab/Qwen3-8B-DFlash-b16 \
  --skip-baseline \
  --batch-requests \
  --attention-backends flashinfer \
  --concurrencies 1,2,4 \
  --questions-per-concurrency-base 32 \
  --max-questions-per-config 128 \
  --context-length 200000 \
  --max-new-tokens 128 \
  --response-json response_kv_retrieval_batch.json \
  --output-md report_kv_retrieval_batch.md

## 常见调整建议

- 只想快速看 DFLASH：加 `--skip-baseline`。
- OOM：先降低 `--concurrencies` 的最大值，再尝试降低 `--mem-fraction-static` 或 `--max-running-requests`。
- prompt 太长被过滤：提高 `--max-input-tokens` 和 `--context-length`，或者换短上下文数据集。
- 长时间无结果：减小 `--max-new-tokens`、`--questions-per-concurrency-base` 或 `--max-questions-per-config`。
- 需要保留报告：传 `--output-md xxx.md`；需要逐题输出和 meta：传 `--response-json xxx.json`。